# Credit Card Fraud (ULB) - preprocessing
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MuhammadDevX/ML-Research-Dataset-Imbalance/blob/main/creditfraud/00_preprocessing/creditfraud_preprocessing.ipynb)

Stateless steps only: label mapping, removing identifiers and duplicates, marking invalid values
as missing, and (for large data) a fixed stratified sample. Everything that is *fitted*
(imputation, scaling, TF-IDF, resampling) happens later inside each training fold.

Writes `data/processed/creditfraud.parquet` and the fixed 5x2 outer folds `data/splits/creditfraud_outer.npz`.
Both are committed, so every experiment notebook (and your teammate) uses identical data and folds.
Raw data is expected in `data/raw/` (not committed; see `data/README.md`).

In [1]:
# ---- Setup: identical in every notebook. Do not edit. ----
import os, sys, subprocess
REPO_URL = "https://github.com/MuhammadDevX/ML-Research-Dataset-Imbalance.git"
BRANCH = "main"
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_ROOT = "/content/ML-Research-Dataset-Imbalance"
    if not os.path.isdir(REPO_ROOT):
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, REPO_ROOT], check=True)
    else:
        subprocess.run(["git", "-C", REPO_ROOT, "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    f"{REPO_ROOT}/requirements.txt"], check=True)
    from google.colab import drive
    drive.mount("/content/drive")
    # Results go to Google Drive so they survive a disconnect.
    os.environ.setdefault("RESULTS_ROOT", "/content/drive/MyDrive/ML-Research-Dataset-Imbalance")
else:
    REPO_ROOT = os.path.abspath(os.getcwd())
    while not os.path.isdir(os.path.join(REPO_ROOT, "common")):
        parent = os.path.dirname(REPO_ROOT)
        if parent == REPO_ROOT:
            raise RuntimeError("Open this notebook from inside the repository.")
        REPO_ROOT = parent

sys.path.insert(0, REPO_ROOT)
os.chdir(REPO_ROOT)
from common.env import check_environment
check_environment()


Environment OK: scikit-learn 1.9.1, imbalanced-learn 0.14.2, xgboost 3.4.1, lightgbm 4.7.0


## Load
Credit Card Fraud Detection (ULB Machine Learning Group, Kaggle): 284,807 European
card transactions over two days in September 2013. `V1`-`V28` are PCA components,
`Time` is seconds since the first transaction, `Amount` is the transaction amount,
`Class` = 1 for fraud.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from common import data, settings
from common.paths import RAW_DIR

raw = pd.read_csv(RAW_DIR / "creditcard.csv")
print(raw.shape)
raw["Class"].value_counts()

(284807, 31)


Class
0    284315
1       492
Name: count, dtype: int64

## Clean (stateless steps only)
* Label: `Class` 1 -> 1 (fraud, minority), 0 -> 0.
* Remove duplicate transactions, then transactions whose features appear with both labels.
* `Time` and `Amount` are kept as they are; scaling happens inside the pipeline.
* **Stratified 100,000-row sample** (seed `MASTER_SEED`), keeping the natural fraud rate.

In [3]:
df = raw.rename(columns={"Class": "y"})
feats = [c for c in df.columns if c != "y"]
print("duplicate rows:", int(df.duplicated().sum()))
df = df.drop_duplicates()
conflict = df.duplicated(subset=feats, keep=False)
print("rows with both labels:", int(conflict.sum()))
df = df[~conflict].reset_index(drop=True)
print(f"after cleaning: {len(df):,} rows, {int(df['y'].sum())} frauds")

CREDIT_N = 100_000
keep, _ = train_test_split(np.arange(len(df)), train_size=CREDIT_N, stratify=df["y"],
                           random_state=settings.MASTER_SEED)
df = df.iloc[np.sort(keep)].reset_index(drop=True)
print(df.shape)
df["y"].value_counts()

duplicate rows: 1081


rows with both labels: 0
after cleaning: 283,726 rows, 473 frauds
(100000, 31)


y
0    99833
1      167
Name: count, dtype: int64

## Save processed data and the 5x2 outer folds

In [4]:
data.save_processed("creditfraud", df)
data.make_outer_splits(df["y"], "creditfraud")
data.summary("creditfraud")

{'dataset': 'creditfraud',
 'n': 100000,
 'd': 30,
 'n_minority': 167,
 'minority_pct': 0.167,
 'IR': 597.8,
 'nan_cells': 0}